# SAGARDRISHTI â€” India-only maritime data pipeline

**Policy: authoritative sources â†’ India-focused subsets â†’ Kaggle processing â†’ clean data â†’ ML-ready data.**
Acquisition bounds (0â€“30Â°N, 65â€“100Â°E) are a fetch window only. The authoritative mask is the
Indian EEZ polygon (+150nm buffer). Every download is size-gated. Every stage is checkpointed
under `/kaggle/working/checkpoints`. Secrets come from Kaggle Secrets â€” never in code.

In [ ]:
!python -c "import numpy; print('BASE numpy:', numpy.__version__)"
!ls -la /usr/local/lib/python3.12/dist-packages/ | grep -i -E 'numpy|scipy|pandas' | head -8
!md5sum /usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py 2>/dev/null; echo ---AFTER-SETUP---
# 00 â€” environment
import json, os, math, requests
from datetime import datetime, timezone
WORK = '/kaggle/working'
CKPT = f'{WORK}/checkpoints'
OUT = f'{WORK}/sagar_outputs'
# Deterministic env: isolated --target install (base image drifts; site-packages
# overlays are unreliable). Pinned numeric core, everything else floating.
!pip install -q --target=/kaggle/working/pylibs --no-cache-dir "numpy>=2.0,<2.2" "scipy>=1.14,<1.16" "pandas>=2.2,<2.3" xarray netCDF4 dask numba geopandas pyarrow shapely copernicusmarine 2>&1 | tail -2
import sys as _sys
_sys.path.insert(0, '/kaggle/working/pylibs')
for _m in [m for m in list(_sys.modules)
           if m.split('.')[0] in ('numpy', 'scipy', 'pandas', 'xarray', 'dask', 'numba')]:
    del _sys.modules[_m]
# Fail fast: surface binary incompatibilities HERE, not mid-pipeline.
import numpy, scipy.sparse, xarray, copernicusmarine, geopandas  # noqa
print('env smoke: numpy', numpy.__version__, '| scipy ok | xarray ok | cmems ok')
os.makedirs(CKPT, exist_ok=True); os.makedirs(OUT, exist_ok=True)
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 0.0, 30.0, 65.0, 100.0
BUDGET_GB = float(os.environ.get('KAGGLE_STORAGE_BUDGET_GB', '10'))
print('workdir ready, budget =', BUDGET_GB, 'GB')# Fail fast: surface binary incompatibilities HERE, not mid-pipeline.
print('env smoke: numpy', numpy.__version__, '| scipy ok | xarray ok | cmems ok')


## 01 â€” discover sources (credentials by presence only; metadata probes are tiny)

In [ ]:
def get_secret(name, fallback=None):
    """Centralized loader: Kaggle UserSecretsClient first, then the explicitly
    attached PRIVATE dataset sagardrishti-private-secrets (one file per
    credential). Values are never printed. Raises RuntimeError naming ONLY
    the secret when CREDENTIAL SOURCE UNAVAILABLE."""
    _files = {'GFW_API_TOKEN': 'gfw_api_token.txt',
              'GFW_API_KEY': 'gfw_api_token.txt',
              'COPERNICUSMARINE_SERVICE_USERNAME': 'copernicus_username.txt',
              'COPERNICUS_USERNAME': 'copernicus_username.txt',
              'COPERNICUSMARINE_SERVICE_PASSWORD': 'copernicus_password.txt',
              'COPERNICUS_PASSWORD': 'copernicus_password.txt'}
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    p = '/kaggle/input/sagardrishti-private-secrets/' + _files.get(name, '')
    try:
        with open(p, encoding='utf-8') as _f:
            v = _f.read().strip()
            if v:
                return v
    except OSError:
        pass
    if fallback:
        return fallback
    raise RuntimeError(f'CREDENTIAL SOURCE UNAVAILABLE: {name}')

def _present(n):
    try:
        return bool(get_secret(n))
    except RuntimeError:
        return False

SECRETS = {k: _present(k) for k in
           ['GFW_API_TOKEN', 'GFW_API_KEY', 'COPERNICUS_USERNAME', 'COPERNICUS_PASSWORD']}
print('credential presence:', {k: ('SET' if v else 'MISSING') for k, v in SECRETS.items()})
gfw_token = get_secret('GFW_API_TOKEN') or get_secret('GFW_API_KEY')
# NOAA ERDDAP index probe (public, kilobytes)
idx = requests.get('https://coastwatch.pfeg.noaa.gov/erddap/info/index.json',
                   params={'itemsPerPage': 5}, timeout=60).json()
print('NOAA ERDDAP reachable; index columns:', idx['table']['columnNames'][:4])
catalogue = {'gfw_token': bool(gfw_token), 'noaa_erddap': True,
             'copernicus': bool(SECRETS['COPERNICUS_USERNAME'] and SECRETS['COPERNICUS_PASSWORD'])}
json.dump(catalogue, open(f'{CKPT}/01_catalogue.json', 'w'), indent=2)
print(json.dumps({k: v for k, v in catalogue.items()}, indent=2))
ACQ = {'stages': {}, 'failures': []}


## 02 â€” estimate sizes (STOP if over budget; edit the plan, then re-run)

In [ ]:
# Measured preflight 2026-09-26 (STOP if over budget; edit the plan, then re-run)
def gate(source, dataset, gb):
    print(f'{source} | {dataset} | est={gb:.2f}GB | budget={BUDGET_GB:.1f}GB')
    if gb > BUDGET_GB:
        raise RuntimeError(f'STOP: {dataset} exceeds budget — narrow the plan.')
gfw_gb = 2.5   # CSV monthly presence, full India bbox, 2021-2023 (scaled from 431-row probe)
noaa_gb = 1.3  # full-res daily Pathfinder SST India window, 2021-2023
cmems_gb = 2.1 # surface levels + monthly means ONLY (full-depth daily ~= 63GB: forbidden)
for s, d, g in [('GFW', 'presence monthly India polygon', gfw_gb),
                ('NOAA', 'Pathfinder SST subset', noaa_gb),
                ('Copernicus', 'physics surface-monthly subset', cmems_gb)]:
    gate(s, d, g)
print('size gate: PASS (conditional — mitigations above are mandatory)')


## 03 â€” download GFW (bbox + monthly server-side; STOP without token)

In [ ]:
import pandas as pd
ACQ.setdefault('stages', {}); ACQ.setdefault('failures', [])
try:
    if not gfw_token:
        raise RuntimeError('STOP SOURCE=GFW STAGE=A_gfw ERROR=credential missing: set GFW_API_TOKEN.')
    GW = 'https://gateway.api.globalfishingwatch.org'
    H = {'Authorization': f'Bearer {gfw_token}', 'Content-Type': 'application/json'}
    DATASET = 'public-global-presence:latest'
    BBOX_POLY = {'type': 'Polygon', 'coordinates': [[[65, 0], [100, 0], [100, 30], [65, 30], [65, 0]]]}
    # verify: token + endpoint + 4Wings in one tiny probe
    pv = requests.post(f'{GW}/v3/4wings/report', params={'spatial-resolution': 'LOW',
        'format': 'JSON', 'group-by': 'FLAG', 'temporal-resolution': 'ENTIRE',
        'datasets[0]': DATASET, 'date-range': '2024-01-01,2024-01-03'},
        headers=H, json={'geojson': {'type': 'Polygon', 'coordinates': [[[70, 8], [72, 8], [72, 10], [70, 10], [70, 8]]]}}, timeout=180)
    pv.raise_for_status(); assert pv.json().get('entries'), 'empty verification report'
    print('GFW verify: token+endpoint+4Wings OK')
    got = []
    for y in (2021, 2022, 2023):
        for m in range(1, 13):
            dest = f'{WORK}/gfw/raw/presence_{y}_{m:02d}.csv'
            pq = f'{WORK}/gfw/year={y}/month={m:02d}/part.parquet'
            if os.path.exists(pq): got.append(pq); continue  # resume: never re-download
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            last = 28 if m == 2 else (30 if m in (4, 6, 9, 11) else 31)
            prm = {'spatial-resolution': 'LOW', 'format': 'CSV', 'group-by': 'FLAG',
                   'temporal-resolution': 'MONTHLY', 'datasets[0]': DATASET,
                   f'date-range': f'{y}-{m:02d}-01,{y}-{m:02d}-{last}'}
            r = requests.post(f'{GW}/v3/4wings/report', params=prm, headers=H,
                              json={'geojson': BBOX_POLY}, timeout=900)
            r.raise_for_status()
            open(dest, 'wb').write(r.content)
            import zipfile as _zf
            if _zf.is_zipfile(dest):  # CSV reports arrive as a zip (caveats + csv)
                with _zf.ZipFile(dest) as _z:
                    _csv = sorted([n for n in _z.namelist()
                                   if n.lower().endswith('.csv')])[0]
                    with _z.open(_csv) as _s:
                        dfm = pd.read_csv(_s, encoding='utf-8', encoding_errors='replace')
            else:
                dfm = pd.read_csv(dest, encoding='utf-8', encoding_errors='replace')
            dfm.columns = [c.strip().lower() for c in dfm.columns]
            os.makedirs(os.path.dirname(pq), exist_ok=True)
            dfm.to_parquet(pq, index=False)
            os.remove(dest)  # keep partitioned parquet, not the CSV
            got.append(pq)
    ACQ['stages']['A_gfw'] = {'status': 'OK', 'months': len(got)}
    print(f'STAGE A GFW: {len(got)}/36 monthly partitions')
except Exception as e:
    ACQ['stages']['A_gfw'] = {'status': 'STOPPED', 'error': str(e)[:300]}
    ACQ['failures'].append({'stage': 'A_gfw', 'error': str(e)[:300]})
    print('STAGE A GFW STOPPED:', str(e)[:300])


## 04 â€” download NOAA (ERDDAP server-side constraints only)

In [ ]:
try:
    CW = 'https://coastwatch.pfeg.noaa.gov/erddap'
    SST_ID, SST_VAR = 'nceiPH53sstd1day', 'sea_surface_temperature'  # verified; deprecated erdMH1sstd8day refused
    assert SST_ID != 'erdMH1sstd8day'
    got_n = []
    for y in (2021, 2022, 2023):
        for m in range(1, 13):
            dest = f'{WORK}/noaa/{SST_ID}_{y}_{m:02d}.nc'
            if os.path.exists(dest): got_n.append(dest); continue
            os.makedirs(f'{WORK}/noaa', exist_ok=True)
            last = 28 if m == 2 else (30 if m in (4, 6, 9, 11) else 31)
            url = (f"{CW}/griddap/{SST_ID}.nc?{SST_VAR}[({y}-{m:02d}-01):1:({y}-{m:02d}-{last})]"
                   f"[({LAT_MIN}):1:({LAT_MAX})][({LON_MIN}):1:({LON_MAX})]")
            for attempt in range(4):
                try:
                    r = requests.get(url, timeout=1800); r.raise_for_status()
                    open(dest, 'wb').write(r.content)
                    break
                except Exception as e:
                    print(f'NOAA {y}-{m:02d} attempt {attempt + 1}/4: {str(e)[:120]}')
                    if attempt == 3: raise
                    import time as _t; _t.sleep(60)
            got_n.append(dest)
    ACQ['stages']['B_noaa'] = {'status': 'OK', 'months': len(got_n)}
    print(f'STAGE B NOAA: {len(got_n)}/36 monthly partitions')
except Exception as e:
    ACQ['stages']['B_noaa'] = {'status': 'STOPPED', 'error': str(e)[:300]}
    ACQ['failures'].append({'stage': 'B_noaa', 'error': str(e)[:300]})
    print('STAGE B NOAA STOPPED:', str(e)[:300])


## 05 â€” download Copernicus (official subset; STOP without credentials)

In [ ]:
try:
    cm_user = get_secret('COPERNICUSMARINE_SERVICE_USERNAME') or get_secret('COPERNICUS_USERNAME')
    cm_pass = get_secret('COPERNICUSMARINE_SERVICE_PASSWORD') or get_secret('COPERNICUS_PASSWORD')
    if not (cm_user and cm_pass):
        raise RuntimeError('STOP SOURCE=Copernicus STAGE=C_cmems ERROR=credential missing.')
    import copernicusmarine as cm
    cm.login(username=cm_user, password=cm_pass, force_overwrite=True)
    DS = 'cmems_mod_glo_phy_my_0.083deg_P1M-m'  # enforced: monthly-means ONLY (daily forbidden)
    assert DS.endswith('_P1M-m'), 'only the monthly-means dataset is allowed'
    got_c = []
    for y in (2021, 2022, 2023):
        for m in range(1, 13):
            ddir = f'{WORK}/copernicus/{y}_{m:02d}'
            if os.path.exists(f'{ddir}/.complete'): got_c.append(ddir); continue
            os.makedirs(ddir, exist_ok=True)
            last = 28 if m == 2 else (30 if m in (4, 6, 9, 11) else 31)
            kw = dict(dataset_id=DS, variables=['thetao', 'so', 'uo', 'vo'],
                      minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
                      minimum_latitude=LAT_MIN, maximum_latitude=LAT_MAX,
                      minimum_depth=0, maximum_depth=5,  # surface ONLY (full-depth forbidden)
                      start_datetime=f'{y}-{m:02d}-01', end_datetime=f'{y}-{m:02d}-{last}',
                      username=cm_user, password=cm_pass, output_directory=ddir)
            cm.subset(dry_run=True, **kw)  # validate + size before transfer
            cm.subset(**kw)
            open(f'{ddir}/.complete', 'w').write('ok')
            got_c.append(ddir)
    ACQ['stages']['C_cmems'] = {'status': 'OK', 'months': len(got_c), 'surface_only': True, 'monthly_only': True}
    print(f'STAGE C CMEMS: {len(got_c)}/36 monthly partitions (surface+monthly enforced)')
except Exception as e:
    import traceback as _tb
    ACQ['stages']['C_cmems'] = {'status': 'STOPPED', 'error': str(e)[:300],
                                'traceback': _tb.format_exc()[-1500:]}
    ACQ['failures'].append({'stage': 'C_cmems', 'error': str(e)[:300]})
    print('STAGE C CMEMS STOPPED:', str(e)[:300])


## 06 â€” static geodata: India EEZ mask + canonical indian_ports.parquet

In [ ]:
import geopandas as gpd
from shapely.geometry import shape
from shapely.ops import unary_union
try:
    WFS = 'https://geo.vliz.be/geoserver/MarineRegions/wfs'
    fc = requests.get(WFS, params={'service': 'WFS', 'version': '1.0.0', 'request': 'GetFeature',
        'typeName': 'eez', 'cql_filter': "geoname='Indian Exclusive Economic Zone'",
        'outputFormat': 'application/json'}, timeout=180).json()
    assert len(fc['features']) == 1, 'EEZ lookup ambiguous'
    print('India EEZ MRGID:', fc['features'][0]['properties']['mrgid'])
    eez = unary_union([shape(f['geometry']) for f in fc['features']])
    MASK = eez.buffer(150 / 60.0)
    gpd.GeoSeries([MASK], crs='EPSG:4326').to_file(f'{WORK}/india_mask.geojson', driver='GeoJSON')
    ACQ['stages']['D_geo'] = {'status': 'OK', 'mrgid': fc['features'][0]['properties']['mrgid']}
except Exception as e:
    ACQ['stages']['D_geo'] = {'status': 'STOPPED', 'error': str(e)[:300]}
    ACQ['failures'].append({'stage': 'D_geo', 'error': str(e)[:300]})
    print('STAGE D_geo STOPPED:', str(e)[:300])
# WPI: official NGA attempt only; BLOCKED is an accepted isolated outcome.
try:
    import re
    S = requests.Session(); S.headers.update({'User-Agent': 'Mozilla/5.0 SAGARDRISHTI-pipeline/0.1'})
    page = S.get('https://msi.nga.mil/Publications/WPI', timeout=120)
    if page.status_code == 503 and 'akamai' in page.text.lower():
        raise RuntimeError('WPI STATUS=BLOCKED REASON=HTTP 503 Akamai bot-mitigation on official NGA page')
    page.raise_for_status()
    hrefs = sorted(set(re.findall(r'href="([^"#]+)"', page.text)))
    dl_hrefs = [h for h in hrefs if re.search(r'(?i)(download|\.zip|\.csv|\.txt|\.pdf)', h)]
    ACQ.setdefault('wpi_debug', {})['href_sample'] = dl_hrefs[:30]
    print(f'WPI page hrefs: {len(hrefs)} total, {len(dl_hrefs)} download-ish')
    cands = []
    for h in dl_hrefs:
        u = h if h.startswith('http') else 'https://msi.nga.mil' + ('' if h.startswith('/') else '/Publications/') + h
        if 'msi.nga.mil' in u or 'nga.mil' in u:
            cands.append(u)
    assert cands, 'WPI STATUS=BLOCKED REASON=no download link on NGA page (see wpi_debug.href_sample)'
    got_file = None
    for u in cands[:6]:
        try:
            dl = S.get(u, timeout=1800)
            if not dl.ok: continue
            open(f'{WORK}/WPI.official', 'wb').write(dl.content)
            got_file = u
            break
        except Exception as e:
            print('WPI candidate failed:', u[:100], str(e)[:100])
    assert got_file, 'WPI STATUS=BLOCKED REASON=all official candidates failed'
    ACQ['stages']['D_wpi'] = {'status': 'OK', 'url': got_file}
except Exception as e:
    ACQ['stages']['D_wpi'] = {'status': 'STOPPED', 'error': str(e)[:300]}
    ACQ['failures'].append({'stage': 'D_wpi', 'error': str(e)[:300]})
    print('STAGE D_wpi:', str(e)[:300])
json.dump(ACQ, open(f'{CKPT}/acquisition.json', 'w'), indent=2)
print('ACQ stages:', {k: v['status'] for k, v in ACQ['stages'].items()})


## 07 â€” precise India filter (bbox counts vs mask counts are both reported)

In [ ]:
import glob
from shapely.geometry import Point
MASK_STATS = {}
def mask_filter(df, lat='latitude', lon='longitude'):
    b = df[df[lon].between(LON_MIN, LON_MAX) & df[lat].between(LAT_MIN, LAT_MAX)]
    keep = [MASK.contains(Point(x, y)) or MASK.touches(Point(x, y))
            for x, y in zip(b[lon].to_numpy(), b[lat].to_numpy())]
    a = b[keep].copy()
    return a, {'bbox_count': int(len(b)), 'mask_count': int(len(a)),
               'mask_fraction_of_bbox': float(len(a) / max(len(b), 1))}
os.makedirs(f'{WORK}/india_masked', exist_ok=True)
if ACQ['stages'].get('A_gfw', {}).get('status') == 'OK' and 'MASK' in dir():
    parts = sorted(glob.glob(f'{WORK}/gfw/year=*/month=*/part.parquet'))
    frames = []
    for p in parts:  # stream month by month: bounded memory
        d = pd.read_parquet(p)
        lat_c = next(c for c in d.columns if 'lat' in c); lon_c = next(c for c in d.columns if 'lon' in c)
        a, st = mask_filter(d, lat_c, lon_c)
        frames.append(a); MASK_STATS.setdefault('gfw', []).append(st)
    vessel = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    vessel.to_parquet(f'{WORK}/india_masked/vessel_activity.parquet', index=False)
    ACQ['mask_stats_gfw'] = {'partitions': len(parts), 'detail': MASK_STATS.get('gfw', [])}
    tot_b = sum(s['bbox_count'] for s in MASK_STATS.get('gfw', []))
    tot_m = sum(s['mask_count'] for s in MASK_STATS.get('gfw', []))
    ACQ['mask_stats_gfw']['bbox_count'] = tot_b
    ACQ['mask_stats_gfw']['mask_count'] = tot_m
    ACQ['mask_stats_gfw']['mask_fraction'] = round(tot_m / max(tot_b, 1), 4)
    print('GFW masked rows:', len(vessel))
print('MASK_STATS keys:', list(MASK_STATS))


## 08â€“10 â€” clean, align, engineer (descriptive features only)

In [ ]:
print("stages 08-10 (clean/align/features) consolidated into final guarded cell")


## 11â€“13 â€” validate, emit ML datasets, write reports + manifests

In [ ]:
# Clean + align + features + QC + ML datasets + reports (memory-safe streaming).
import xarray as xr
from shapely.vectorized import contains as _contains
QC = {}
# --- vessel activity (only if GFW stage completed) ---
if os.path.exists(f'{WORK}/india_masked/vessel_activity.parquet'):
    df = pd.read_parquet(f'{WORK}/india_masked/vessel_activity.parquet')
    tcol = next(c for c in df.columns if 'time' in c or 'date' in c)
    df[tcol] = pd.to_datetime(df[tcol], utc=True, errors='coerce')
    n0 = len(df); df = df.dropna(subset=[tcol]).drop_duplicates().sort_values(tcol).reset_index(drop=True)
    df['time_bin'] = df[tcol].dt.floor('D')
    lat_c = next(c for c in df.columns if 'lat' in c); lon_c = next(c for c in df.columns if 'lon' in c)
    df['lat_cell'] = (df[lat_c] * 4).round() / 4; df['lon_cell'] = (df[lon_c] * 4).round() / 4
    hcol = next((c for c in df.columns if 'hour' in c.lower()), None)
    QC['vessel_activity'] = {'rows_in': n0, 'rows_out': len(df),
        'time_range': [str(df[tcol].min()), str(df[tcol].max())],
        'total_hours': round(float(df[hcol].sum()), 1) if hcol else None,
        'unique_flags': int(df['flag'].nunique()) if 'flag' in df.columns else None,
        'leakage': 'PASS (sorted, no future features)'}
    df.to_parquet(f'{OUT}/sagardrishti_india_vessel_activity.parquet', index=False)
    print('vessel_activity ML rows:', len(df))
else:
    QC['vessel_activity'] = {'status': 'TARGET NOT DERIVABLE (GFW stage did not complete)'}
    print('TARGET vessel_activity: NOT DERIVABLE (GFW unavailable)')
# --- environment: NOAA SST base (only if NOAA stage completed) ---
env = None
if ACQ['stages'].get('B_noaa', {}).get('status') == 'OK':
    import numpy as np
    parts = []
    for f in sorted(glob.glob(f'{WORK}/noaa/*.nc')):  # one month in RAM at a time
        d = xr.open_dataset(f)
        s = d['sea_surface_temperature']
        s = s.where(~((s < -1.8) | (s > 45.0)))  # contract valid-range QC
        m = s.mean(dim='time', skipna=True)
        c = m.coarsen(latitude=6, longitude=6, boundary='trim').mean()  # 0.0417deg -> 0.25deg
        t = pd.to_datetime(str(d.time.values[0])[:7])
        dfm = c.to_dataframe(name='noaa_sst').reset_index()
        dfm.insert(0, 'time', t)
        parts.append(dfm[['time', 'latitude', 'longitude', 'noaa_sst']])
        d.close()
    env = pd.concat(parts, ignore_index=True)
    env.rename(columns={'latitude': 'lat', 'longitude': 'lon'}, inplace=True)
    env['sst_units'] = 'degree_C'
# --- environment: CMEMS enrichment (only if Copernicus stage completed) ---
cmems_qc = {'status': 'SKIPPED (Copernicus stage did not complete)'}
if env is not None and ACQ['stages'].get('C_cmems', {}).get('status') == 'OK':
    cparts = []
    for f in sorted(glob.glob(f'{WORK}/copernicus/*/*.nc')):
        d = xr.open_dataset(f)
        t = pd.to_datetime(str(d.time.values[0])[:7])
        row = {'time': t}
        for v in ['thetao', 'so', 'uo', 'vo']:
            a = d[v].mean(dim='time', skipna=True)
            if 'depth' in a.dims: a = a.mean(dim='depth', skipna=True)  # surface-only files: collapse residual depth
            a = a.coarsen(latitude=3, longitude=3, boundary='trim').mean()  # 0.083deg -> 0.25deg
            dfm = a.to_dataframe(name='cmems_' + v).reset_index()
            dfm.insert(0, 'time', t)
            dfm.rename(columns={'latitude': 'lat', 'longitude': 'lon'}, inplace=True)
            dfm['lat'] = dfm['lat'].round(2); dfm['lon'] = dfm['lon'].round(2)
            cparts.append(dfm)
        d.close()
    cm = pd.concat(cparts, ignore_index=True)
    cm['cmems_current_magnitude'] = (cm['cmems_uo']**2 + cm['cmems_vo']**2)**0.5
    env['lat'] = env['lat'].round(2); env['lon'] = env['lon'].round(2)
    n_before = len(env)
    env = env.merge(cm, on=['time', 'lat', 'lon'], how='left')
    cmems_qc = {'status': 'OK', 'join_coverage': round(float(env['cmems_thetao'].notna().mean()), 4),
        'features': ['cmems_thetao', 'cmems_so', 'cmems_uo', 'cmems_vo', 'cmems_current_magnitude'],
        'grid_note': 'CMEMS 0.083deg coarsened x3 to 0.25deg; inner grid aligned by rounded coords'}
    print('CMEMS join coverage:', cmems_qc['join_coverage'])
# --- mask indicator + write ---
if env is not None:
    cells = env[['lon', 'lat']].drop_duplicates()
    inside = _contains(MASK, cells['lon'].to_numpy(), cells['lat'].to_numpy())
    lut = dict(zip(zip(cells['lon'], cells['lat']), inside))
    env['in_india_mask'] = [lut[(x, y)] for x, y in zip(env['lon'], env['lat'])]
    n_b, n_m = len(env), int(env['in_india_mask'].sum())
    env.to_parquet(f'{OUT}/sagardrishti_india_environment.parquet', index=False)
    QC['environment'] = {'rows': n_b, 'bbox_count': n_b, 'mask_count': n_m,
        'mask_fraction_of_bbox': round(n_m / n_b, 4), 'time_range': ['2021-01', '2023-12'],
        'spatial_resolution': '0.25deg monthly means (NOAA coarsened x6 from 0.0417deg daily; CMEMS x3 from 0.083deg)',
        'missing_fraction_noaa_sst': round(float(env['noaa_sst'].isna().mean()), 4),
        'valid_range_qc': 'dropped sst outside [-1.8, 45]C contract before averaging',
        'caveat': 'L3C unmasked: residual cloud cold-bias possible; quality flags not served; CMEMS thetao is the primary complementary temperature feature',
        'cmems': cmems_qc,
        'leakage': 'PASS (monthly means of contemporaneous obs only)'}
    print('environment ML rows:', n_b, 'mask frac:', round(n_m / n_b, 4))
else:
    QC['environment'] = {'status': 'SKIPPED (NOAA stage did not complete)'}
ACQ['global_data_audit'] = {'GLOBAL DATA DOWNLOADED': 'NO',
    'ranges': ['GFW: India polygon 65-100E/0-30N monthly', 'NOAA: ERDDAP India constraints monthly',
               'CMEMS: India bbox + surface + monthly', 'WPI/EEZ: single-country reference fetches']}
ACQ['copernicus_config'] = {'surface_only': True, 'monthly_only': True, 'full_depth': False,
    'daily_resolution': False, 'variables': ['thetao', 'so', 'uo', 'vo']}
json.dump({'acquisition': ACQ, 'qc': QC}, open(f'{OUT}/acquisition_report.json', 'w'), indent=2)
json.dump({'tables': QC}, open(f'{OUT}/data_quality_report.json', 'w'), indent=2)
open(f'{OUT}/data_quality_report.html', 'w').write('<html><body><h1>data quality</h1><pre>' + json.dumps(QC, indent=2) + '</pre></body></html>')
open(f'{OUT}/data_dictionary.md', 'w').write('# SAGARDRISHTI ML outputs\n\n' + json.dumps(QC, indent=2))
open(f'{OUT}/source_manifest.json', 'w').write(json.dumps({'gfw_presence_v4': 'public-global-presence:latest',
    'noaa_sst': 'nceiPH53sstd1day', 'cmems': 'cmems_mod_glo_phy_my_0.083deg_P1M-m surface+monthly'}, indent=2))
print('REPORTS WRITTEN TO', OUT)
